In [1]:
import pandas as pd
import numpy as np
import time

from src.features import load_data

In [2]:
# consDF, testDF, acctDF, trxnDF, cat_map = load_data.load_data()

# Pipeline

## feature creations

In [3]:
all_feats = pd.read_parquet('src/all_feats_do_not_push.parquet')
# dq = all_feats[all_feats['DQ_TARGET'] == 0]
# not_dq = all_feats[all_feats['DQ_TARGET'] != 0]

consumer_ids = all_feats['prism_consumer_id']
y = all_feats['DQ_TARGET']
X = all_feats.drop(columns=['prism_consumer_id','DQ_TARGET'])
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

X_train.head()

,avg_monthly_neg_count,total_neg_amount,has_neg,avg_balance_x,avg_monthly_income_x,income_std,total_balance,avg_balance_y,max_balance,min_balance,...,avg_cat47_spend_6m,med_cat47_spend_6m,sd_cat47_spend_6m,min_cat47_spend_6m,max_cat47_spend_6m,avg_cat47_to_income_ratio_6m,med_cat47_to_income_ratio_6m,sd_cat47_to_income_ratio_6m,min_cat47_to_income_ratio_6m,max_cat47_to_income_ratio_6m
457,2.500000,377.54,1.0,701.900,2195.966250,687.667758,2105.70,701.900,1500.00,300.13,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3833,1.666667,128.44,1.0,4693.930,5403.565000,1351.914717,9387.86,4693.930,9375.26,12.60,...,4.483801,3.255112,4.377882,1.486178,8.901168,0.000194,0.000189,0.000109,0.000121,0.000273
479,1.000000,33.00,1.0,529.470,595.518750,642.167716,1058.94,529.470,921.67,137.27,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1114,2.000000,10.81,1.0,1128.475,3015.621111,791.739564,2256.95,1128.475,2231.95,25.00,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
798,1.000000,2.50,1.0,674.430,2300.670000,2247.456379,1105.74,674.430,989.66,158.62,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


## feat engineering

In [4]:
# pca
from sklearn.decomposition import PCA
pca = PCA(n_components = 100)

In [5]:
# standard scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [6]:
# normalize
from sklearn.preprocessing import Normalizer
norm = Normalizer()

## Resampling for Uneven Dist

In [7]:
# # smote
# print("Before OverSampling, counts of label '1': {}".format(sum(y_train == 1)))
# print("Before OverSampling, counts of label '0': {} \n".format(sum(y_train == 0)))

# # import SMOTE module from imblearn library
# from imblearn.over_sampling import SMOTE
# sm = SMOTE(random_state = 2)
# X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

# print('After OverSampling, the shape of train_X: {}'.format(X_train_res.shape))
# print('After OverSampling, the shape of train_y: {} \n'.format(y_train_res.shape))

# print("After OverSampling, counts of label '1': {}".format(sum(y_train_res == 1)))
# print("After OverSampling, counts of label '0': {}".format(sum(y_train_res == 0)))

In [8]:
# # near miss algorithm 
# print("Before Undersampling, counts of label '1': {}".format(sum(y_train == 1)))
# print("Before Undersampling, counts of label '0': {} \n".format(sum(y_train == 0)))

# # apply near miss
# from imblearn.under_sampling import NearMiss
# nr = NearMiss()

# X_train_miss, y_train_miss = nr.fit_resample(X_train, y_train)

# print('After Undersampling, the shape of train_X: {}'.format(X_train_miss.shape))
# print('After Undersampling, the shape of train_y: {} \n'.format(y_train_miss.shape))

# print("After Undersampling, counts of label '1': {}".format(sum(y_train_miss == 1)))
# print("After Undersampling, counts of label '0': {}".format(sum(y_train_miss == 0)))

In [9]:
# def uneven_dist_resampling(X, y, resample = 'none'):
#     if resample == 'none':
#         return X
#     elif resample == 'smote':
#         return X_train_res, y_train_res
#     elif resample == 'near miss':
#         return X_train_miss, y_train_miss

In [10]:
from src.imbalance.uneven_resampling import uneven_dist_resampling

X_train, y_train = uneven_dist_resampling(X_train, y_train, resample = 'none')
X_sm, y_sm = uneven_dist_resampling(X_train, y_train, resample = 'smote')
X_miss, y_miss = uneven_dist_resampling(X_train, y_train, resample = 'near miss')

Before SMOTE, counts of label '1': 526
Before SMOTE, counts of label '0': 4137 

After SMOTE, the shape of X: (8274, 1170)
After SMOTE, the shape of y: (8274,) 

After SMOTE, counts of label '1': 4137
After SMOTE, counts of label '0': 4137
Before near miss, counts of label '1': 526
Before near miss, counts of label '0': 4137 

After near miss, the shape of X: (1052, 1170)
After near miss, the shape of y: (1052,) 

After near miss, counts of label '1': 526
After near miss, counts of label '0': 526


## Models

In [11]:
X_train.head()

,avg_monthly_neg_count,total_neg_amount,has_neg,avg_balance_x,avg_monthly_income_x,income_std,total_balance,avg_balance_y,max_balance,min_balance,...,avg_cat47_spend_6m,med_cat47_spend_6m,sd_cat47_spend_6m,min_cat47_spend_6m,max_cat47_spend_6m,avg_cat47_to_income_ratio_6m,med_cat47_to_income_ratio_6m,sd_cat47_to_income_ratio_6m,min_cat47_to_income_ratio_6m,max_cat47_to_income_ratio_6m
457,2.500000,377.54,1.0,701.900,2195.966250,687.667758,2105.70,701.900,1500.00,300.13,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3833,1.666667,128.44,1.0,4693.930,5403.565000,1351.914717,9387.86,4693.930,9375.26,12.60,...,4.483801,3.255112,4.377882,1.486178,8.901168,0.000194,0.000189,0.000109,0.000121,0.000273
479,1.000000,33.00,1.0,529.470,595.518750,642.167716,1058.94,529.470,921.67,137.27,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1114,2.000000,10.81,1.0,1128.475,3015.621111,791.739564,2256.95,1128.475,2231.95,25.00,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
798,1.000000,2.50,1.0,674.430,2300.670000,2247.456379,1105.74,674.430,989.66,158.62,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [12]:
y_train.head()

457     0.0
3833    0.0
479     0.0
1114    0.0
798     0.0
Name: DQ_TARGET, dtype: float64

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold

In [14]:
# logistic
logreg = LogisticRegression(
    solver="saga",
    penalty="l1",
    C=0.1,
    max_iter=5000,
    tol=1e-3,
    random_state=0
)
# logreg.fit(X_train, y_train)

# preds = model.predict_proba(X_train)[:, 1]

# score = roc_auc_score(y_train, preds)

# # auc_df = pd.concat([auc_df, pd.DataFrame([{"feature": col, "auc_roc_mean": score}])],ignore_index=True)
# # print(col)
# print(len(X_train)) #number of consumers
# print(score)

## pipeline

### Logistic Regression

In [15]:
from sklearn.feature_selection import VarianceThreshold
pipe = Pipeline([("vt", VarianceThreshold(1e-6)),
                 ('scaler', scaler),  
                 ('pca', pca), 
                 ('normalize', norm),
                  ('logreg', logreg)
         ])

param_grid = [
    # # lbfgs + l2 only
    # {
    #     "pca__n_components": [350, 400, 500, 600],
    #     "logreg__solver": ["lbfgs"],
    #     "logreg__penalty": ["l2"],
    #     "logreg__C": [0.01, 0.1, 1.0],
    #     "logreg__max_iter": [1500, 2000, 2500],
    #     "logreg__tol": [1e-4, 1e-3],
    # },
    # saga + l1/l2
    {
        # "pca__n_components": [300, 350, 400], #prev:500, 350
        "logreg__solver": ["saga"],
        "logreg__penalty": ["l1", "l2"],
        "logreg__C": [0.01, 0.1, 1.0],
        "logreg__max_iter": [2000], #prev: 2000, 1700, 2000
        "logreg__tol": [1e-4, 1e-3],
    },
]

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)

grid = GridSearchCV(
    pipe,
    param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose = 1,
    error_score="raise"
)

t0 = time.perf_counter()
grid.fit(X_train, y_train)
grid_fit_s = time.perf_counter() - t0

best = grid.best_estimator_

print("best estimator:", grid.best_estimator_)
print("best params:", grid.best_params_)

print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

print(f"\n[Latency] GridSearchCV.fit total: {grid_fit_s:.2f} s "
      f"({grid_fit_s/60:.2f} min)")

# --- Best estimator refit time (optional but useful) ---
# Note: GridSearchCV already refits best_estimator_ by default, but we time it explicitly.
t0 = time.perf_counter()
best.fit(X_train, y_train)
best_fit_s = time.perf_counter() - t0

# --- Train scoring latency ---
t0 = time.perf_counter()
preds_train = best.predict_proba(X_train)[:, 1]
pred_train_s = time.perf_counter() - t0

t0 = time.perf_counter()
train_auc = roc_auc_score(y_train, preds_train)
train_auc_s = time.perf_counter() - t0

# --- Test scoring latency ---
t0 = time.perf_counter()
preds_test = best.predict_proba(X_test)[:, 1]
pred_test_s = time.perf_counter() - t0

t0 = time.perf_counter()
test_auc = roc_auc_score(y_test, preds_test)
test_auc_s = time.perf_counter() - t0

print("\nPerformance")
print(f"best CV auc: {grid.best_score_}")
print(f"train auc-roc: {train_auc:.6f}")
print(f"test  auc-roc: {test_auc:.6f}")

print("\n[Latency] Best-estimator timings")
print(f"best.fit(X_train,y_train):      {best_fit_s:.4f} s")
print(f"predict_proba(X_train):         {pred_train_s:.4f} s")
print(f"roc_auc_score(train):           {train_auc_s:.6f} s")
print(f"predict_proba(X_test):          {pred_test_s:.4f} s")
print(f"roc_auc_score(test):            {test_auc_s:.6f} s")

# Optional: per-row latency (useful for “serving” style reporting)
ntr = len(y_train)
nte = len(y_test)
print("\n[Latency] Per-row (approx)")
print(f"train predict_proba per row: {pred_train_s/ntr*1e3:.6f} ms")
print(f"test  predict_proba per row: {pred_test_s/nte*1e3:.6f} ms")

# best params: {'logreg__C': 1.0, 'logreg__max_iter': 2000, 'logreg__penalty': 'l2', 'logreg__solver': 'saga', 'logreg__tol': 0.0001, 'pca__n_components': 350}
# best CV auc: 0.715231153074429
# train auc-roc:  0.7816872975962226
# test auc-roc:  0.7672276059787072

# best params: {'logreg__C': 1.0, 'logreg__max_iter': 2000, 'logreg__penalty': 'l2', 'logreg__solver': 'saga', 'logreg__tol': 0.001, 'pca__n_components': 350}
# best CV auc: 0.7151557716708414
# train auc-roc:  0.7815749892667357
# test auc-roc:  0.7674142539308059

Fitting 5 folds for each of 12 candidates, totalling 60 fits
best estimator: Pipeline(steps=[('vt', VarianceThreshold(threshold=1e-06)),
                ('scaler', StandardScaler()), ('pca', PCA(n_components=100)),
                ('normalize', Normalizer()),
                ('logreg',
                 LogisticRegression(max_iter=2000, solver='saga', tol=0.001))])
best params: {'logreg__C': 1.0, 'logreg__max_iter': 2000, 'logreg__penalty': 'l2', 'logreg__solver': 'saga', 'logreg__tol': 0.001}
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

[Latency] GridSearchCV.fit total: 29.39 s (0.49 min)

Performance
best CV auc: 0.6968999447692511
train auc-roc: 0.736272
test  auc-roc: 0.731309

[Latency] Best-estimator timings
best.fit(X_train,y_train):      20.9258 s
predict_proba(X_train):         0.4252 s
roc_auc_score(train):           0.004402 s
predict_proba(X_test):          0.1987 s
roc_auc_score(test):            0.095719 s

[Latency] Per-row (approx)
train predict_proba per row:

In [16]:
# from sklearn.model_selection import StratifiedKFold, cross_val_score

# aucs = cross_val_score(
#     grid.best_estimator_,
#     X_train,
#     y_train,
#     scoring="roc_auc",
#     cv=cv
# )

# aucs, aucs.mean(), aucs.std()

In [17]:
# # subgroup
# mask_0_train = (y_train == 0)
# mask_1_train = (y_train == 1)

# X_train_0 = X_train.loc[mask_0_train]
# y_train_0 = y_train.loc[mask_0_train]

# X_train_1 = X_train.loc[mask_1_train]
# y_train_1 = y_train.loc[mask_1_train]

# mask_0_test = (y_test == 0)
# mask_1_test = (y_test == 1)

# X_test_0 = X_test.loc[mask_0_test]
# y_test_0 = y_test.loc[mask_0_test]

# X_test_1 = X_test.loc[mask_1_test]
# y_test_1 = y_test.loc[mask_1_test]

# preds_train_0 = grid.best_estimator_.predict_proba(X_train_0)[:, 1]
# print(f'train_0 auc-roc: {roc_auc_score(y_train_0, preds_train_0)}')
# preds_test_0 = grid.best_estimator_.predict_proba(X_test_0)[:, 1]
# print(f'test_0 auc-roc: {roc_auc_score(y_test_0, preds_test_0)}')

# preds_train_1 = grid.best_estimator_.predict_proba(X_train_1)[:, 1]
# print(f'train_1 auc-roc: {roc_auc_score(y_train_1, preds_train_1)}')
# preds_test_1 = grid.best_estimator_.predict_proba(X_test_1)[:, 1]
# print(f'test_1 auc-roc: {roc_auc_score(y_test_1, preds_test_1)}')

In [18]:
pipe = Pipeline([('scaler', scaler), 
                 ('pca', pca), 
                 ('logreg', logreg)
                ])

param_grid = [
    # # lbfgs + l2 only
    # {
    #     "pca__n_components": [50, 100, 200],
    #     "logreg__solver": ["lbfgs"],
    #     "logreg__penalty": ["l2"],
    #     "logreg__C": [0.001, 0.01, 0.1],
    #     "logreg__max_iter": [2000, 5000],
    #     # "logreg__tol": [1e-4, 1e-3],
    # },
    # saga + l1/l2
    {
        "pca__n_components": [400, 500, 600], #prev:200, 500
        "logreg__solver": ["saga"],
        "logreg__penalty": ["l1", "l2"],
        "logreg__C": [0.05, 0.1, 1], #prev: 0.1
        "logreg__max_iter": [8000, 10000, 12000], #prev: 10000, 10000
        # "logreg__tol": [1e-4, 1e-3],
    },
]

grid = GridSearchCV(
    pipe,
    param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1, 
    error_score="raise"
)

t0 = time.perf_counter()
grid.fit(X_train, y_train)
grid_fit_s = time.perf_counter() - t0

best = grid.best_estimator_

print("best estimator:", grid.best_estimator_)
print("best params:", grid.best_params_)

print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

print(f"\n[Latency] GridSearchCV.fit total: {grid_fit_s:.2f} s "
      f"({grid_fit_s/60:.2f} min)")

# --- Best estimator refit time (optional but useful) ---
# Note: GridSearchCV already refits best_estimator_ by default, but we time it explicitly.
t0 = time.perf_counter()
best.fit(X_train, y_train)
best_fit_s = time.perf_counter() - t0

# --- Train scoring latency ---
t0 = time.perf_counter()
preds_train = best.predict_proba(X_train)[:, 1]
pred_train_s = time.perf_counter() - t0

t0 = time.perf_counter()
train_auc = roc_auc_score(y_train, preds_train)
train_auc_s = time.perf_counter() - t0

# --- Test scoring latency ---
t0 = time.perf_counter()
preds_test = best.predict_proba(X_test)[:, 1]
pred_test_s = time.perf_counter() - t0

t0 = time.perf_counter()
test_auc = roc_auc_score(y_test, preds_test)
test_auc_s = time.perf_counter() - t0

print("\nPerformance")
print(f"best CV auc: {grid.best_score_}")
print(f"train auc-roc: {train_auc:.6f}")
print(f"test  auc-roc: {test_auc:.6f}")

print("\n[Latency] Best-estimator timings")
print(f"best.fit(X_train,y_train):      {best_fit_s:.4f} s")
print(f"predict_proba(X_train):         {pred_train_s:.4f} s")
print(f"roc_auc_score(train):           {train_auc_s:.6f} s")
print(f"predict_proba(X_test):          {pred_test_s:.4f} s")
print(f"roc_auc_score(test):            {test_auc_s:.6f} s")

# Optional: per-row latency (useful for “serving” style reporting)
ntr = len(y_train)
nte = len(y_test)
print("\n[Latency] Per-row (approx)")
print(f"train predict_proba per row: {pred_train_s/ntr*1e3:.6f} ms")
print(f"test  predict_proba per row: {pred_test_s/nte*1e3:.6f} ms")

# best params: {'logreg__C': 0.1, 'logreg__max_iter': 10000, 'logreg__penalty': 'l1', 'logreg__solver': 'saga', 'pca__n_components': 500}
# best CV auc: 0.7136976317599888
# train auc-roc:  0.7660844956692583
# test auc-roc:  0.746927774708456

Fitting 5 folds for each of 54 candidates, totalling 270 fits
best estimator: Pipeline(steps=[('scaler', StandardScaler()), ('pca', PCA(n_components=500)),
                ('logreg',
                 LogisticRegression(C=0.05, max_iter=8000, solver='saga',
                                    tol=0.001))])
best params: {'logreg__C': 0.05, 'logreg__max_iter': 8000, 'logreg__penalty': 'l2', 'logreg__solver': 'saga', 'pca__n_components': 500}
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

[Latency] GridSearchCV.fit total: 482.14 s (8.04 min)

Performance
best CV auc: 0.7057305978711119
train auc-roc: 0.785338
test  auc-roc: 0.739881

[Latency] Best-estimator timings
best.fit(X_train,y_train):      69.6368 s
predict_proba(X_train):         1.0917 s
roc_auc_score(train):           0.098105 s
predict_proba(X_test):          0.5975 s
roc_auc_score(test):            0.099175 s

[Latency] Per-row (approx)
train predict_proba per row: 0.234127 ms
test  predict_proba per row: 0.384230 ms


In [19]:
pipe = Pipeline([('scaler', scaler),  
                 ('pca', pca), 
                 ('normalize', norm),
                  ('logreg', logreg)
         ])

param_grid = [
    # # lbfgs + l2 only
    # {
    #     "pca__n_components": [150, 200, 300, 500],
    #     "logreg__solver": ["lbfgs"],
    #     "logreg__penalty": ["l2"],
    #     "logreg__C": [0.01, 0.1, 1.0],
    #     "logreg__max_iter": [1500, 2000, 2500],
    #     "logreg__tol": [1e-4, 1e-3],
    # },
    # saga + l1/l2
    {
        "pca__n_components": [700, 800, 1000],#prev: 800
        "logreg__solver": ["saga"],
        "logreg__penalty": ["l1", "l2"],
        "logreg__C": [1.0],
        "logreg__max_iter": [1500, 2000, 2200], #prev: 2200
        "logreg__tol": [1e-3],
    },
]

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)

grid = GridSearchCV(
    pipe,
    param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    error_score="raise"
)

t0 = time.perf_counter()
grid.fit(X_train, y_train)
grid_fit_s = time.perf_counter() - t0

best = grid.best_estimator_

print("best estimator:", grid.best_estimator_)
print("best params:", grid.best_params_)

print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

print(f"\n[Latency] GridSearchCV.fit total: {grid_fit_s:.2f} s "
      f"({grid_fit_s/60:.2f} min)")

# --- Best estimator refit time (optional but useful) ---
# Note: GridSearchCV already refits best_estimator_ by default, but we time it explicitly.
t0 = time.perf_counter()
best.fit(X_train, y_train)
best_fit_s = time.perf_counter() - t0

# --- Train scoring latency ---
t0 = time.perf_counter()
preds_train = best.predict_proba(X_train)[:, 1]
pred_train_s = time.perf_counter() - t0

t0 = time.perf_counter()
train_auc = roc_auc_score(y_train, preds_train)
train_auc_s = time.perf_counter() - t0

# --- Test scoring latency ---
t0 = time.perf_counter()
preds_test = best.predict_proba(X_test)[:, 1]
pred_test_s = time.perf_counter() - t0

t0 = time.perf_counter()
test_auc = roc_auc_score(y_test, preds_test)
test_auc_s = time.perf_counter() - t0

print("\nPerformance")
print(f"best CV auc: {grid.best_score_}")
print(f"train auc-roc: {train_auc:.6f}")
print(f"test  auc-roc: {test_auc:.6f}")

print("\n[Latency] Best-estimator timings")
print(f"best.fit(X_train,y_train):      {best_fit_s:.4f} s")
print(f"predict_proba(X_train):         {pred_train_s:.4f} s")
print(f"roc_auc_score(train):           {train_auc_s:.6f} s")
print(f"predict_proba(X_test):          {pred_test_s:.4f} s")
print(f"roc_auc_score(test):            {test_auc_s:.6f} s")

# Optional: per-row latency (useful for “serving” style reporting)
ntr = len(y_train)
nte = len(y_test)
print("\n[Latency] Per-row (approx)")
print(f"train predict_proba per row: {pred_train_s/ntr*1e3:.6f} ms")
print(f"test  predict_proba per row: {pred_test_s/nte*1e3:.6f} ms")

# best params: {'logreg__C': 1.0, 'logreg__max_iter': 2200, 'logreg__penalty': 'l1', 'logreg__solver': 'saga', 'logreg__tol': 0.001, 'pca__n_components': 800}
# best CV auc: 0.8173323572949929
# train auc-roc:  0.8474395230311801
# test auc-roc:  0.7377446954652014

Fitting 5 folds for each of 18 candidates, totalling 90 fits
best estimator: Pipeline(steps=[('scaler', StandardScaler()), ('pca', PCA(n_components=700)),
                ('normalize', Normalizer()),
                ('logreg',
                 LogisticRegression(max_iter=2000, penalty='l1', solver='saga',
                                    tol=0.001))])
best params: {'logreg__C': 1.0, 'logreg__max_iter': 2000, 'logreg__penalty': 'l1', 'logreg__solver': 'saga', 'logreg__tol': 0.001, 'pca__n_components': 700}
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

[Latency] GridSearchCV.fit total: 201.73 s (3.36 min)

Performance
best CV auc: 0.7110569241164423
train auc-roc: 0.768434
test  auc-roc: 0.739088

[Latency] Best-estimator timings
best.fit(X_train,y_train):      82.9883 s
predict_proba(X_train):         1.2762 s
roc_auc_score(train):           0.003467 s
predict_proba(X_test):          0.9933 s
roc_auc_score(test):            0.002302 s

[Latency] Per-row (approx)
train predi

In [20]:
# import time
# from sklearn.ensemble import RandomForestClassifier

# rf_test = RandomForestClassifier(
#     n_estimators=400,     # pick a typical value from your grid
#     max_depth=None,       # typical
#     min_samples_leaf=1,   # typical
#     max_features="sqrt",
#     n_jobs=1,
#     random_state=0
# )

# t0 = time.time()
# rf_test.fit(X_train, y_train)
# t_fit = time.time() - t0
# print("one fit (seconds):", t_fit)

# # estimate wall time for 1080 fits on 5 cores
# est_seconds = t_fit * (1080 / 5)
# print("estimated total (hours):", est_seconds / 3600)

In [23]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, mutual_info_classif


rf = RandomForestClassifier(random_state=0)
imputer = SimpleImputer(strategy="mean", add_indicator=True)
# skb = SelectKBest(mutual_info_classif, k=300)


pipe = Pipeline([#("imputer", imputer),
                 #("skb", skb),
                 ('rf', rf)
                ])


param_grid = {
    "rf__n_estimators": [350, 400, 450],
    "rf__max_features": ["sqrt", 0.1],
    "rf__max_depth": [15, 20],
    "rf__min_samples_split": [10, 20],
    "rf__min_samples_leaf": [20, 50],
    "rf__class_weight": ["balanced"],
}


cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)

grid = GridSearchCV(
    pipe,
    param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

t0 = time.perf_counter()
grid.fit(X_sm, y_sm)
grid_fit_s = time.perf_counter() - t0

best = grid.best_estimator_

print("best estimator:", grid.best_estimator_)
print("best params:", grid.best_params_)

print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

print(f"\n[Latency] GridSearchCV.fit total: {grid_fit_s:.2f} s "
      f"({grid_fit_s/60:.2f} min)")

# --- Best estimator refit time (optional but useful) ---
# Note: GridSearchCV already refits best_estimator_ by default, but we time it explicitly.
t0 = time.perf_counter()
best.fit(X_sm, y_sm)
best_fit_s = time.perf_counter() - t0

# --- Train scoring latency ---
t0 = time.perf_counter()
preds_train = best.predict_proba(X_sm)[:, 1]
pred_train_s = time.perf_counter() - t0

t0 = time.perf_counter()
train_auc = roc_auc_score(y_sm, preds_train)
train_auc_s = time.perf_counter() - t0

# --- Test scoring latency ---
t0 = time.perf_counter()
preds_test = best.predict_proba(X_test)[:, 1]
pred_test_s = time.perf_counter() - t0

t0 = time.perf_counter()
test_auc = roc_auc_score(y_test, preds_test)
test_auc_s = time.perf_counter() - t0

print("\nPerformance")
print(f"best CV auc: {grid.best_score_}")
print(f"train auc-roc: {train_auc:.6f}")
print(f"test  auc-roc: {test_auc:.6f}")

print("\n[Latency] Best-estimator timings")
print(f"best.fit(X_train,y_train):      {best_fit_s:.4f} s")
print(f"predict_proba(X_train):         {pred_train_s:.4f} s")
print(f"roc_auc_score(train):           {train_auc_s:.6f} s")
print(f"predict_proba(X_test):          {pred_test_s:.4f} s")
print(f"roc_auc_score(test):            {test_auc_s:.6f} s")

# Optional: per-row latency (useful for “serving” style reporting)
ntr = len(y_sm)
nte = len(y_test)
print("\n[Latency] Per-row (approx)")
print(f"train predict_proba per row: {pred_train_s/ntr*1e3:.6f} ms")
print(f"test  predict_proba per row: {pred_test_s/nte*1e3:.6f} ms")

# best params: {'rf__class_weight': 'balanced', 'rf__max_depth': 20, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 10, 'rf__min_samples_split': 2, 'rf__n_estimators': 500}
# best CV auc: 0.7226789743202131
# train auc-roc:  0.8906017357086465
# test auc-roc:  0.7368301204999179

# best params: {'rf__class_weight': 'balanced', 'rf__max_depth': 20, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 5, 'rf__min_samples_split': 2, 'rf__n_estimators': 600}
# best CV auc: 0.730654200845845
# train auc-roc:  0.9997834392127612
# test auc-roc:  0.7587873855848054

Fitting 3 folds for each of 48 candidates, totalling 144 fits
best estimator: Pipeline(steps=[('rf',
                 RandomForestClassifier(class_weight='balanced', max_depth=20,
                                        max_features=0.1, min_samples_leaf=20,
                                        min_samples_split=10, n_estimators=450,
                                        random_state=0))])
best params: {'rf__class_weight': 'balanced', 'rf__max_depth': 20, 'rf__max_features': 0.1, 'rf__min_samples_leaf': 20, 'rf__min_samples_split': 10, 'rf__n_estimators': 450}
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

[Latency] GridSearchCV.fit total: 1101.12 s (18.35 min)

Performance
best CV auc: 0.9531420494194225
train auc-roc: 0.990980
test  auc-roc: 0.741847

[Latency] Best-estimator timings
best.fit(X_train,y_train):      159.8381 s
predict_proba(X_train):         0.5121 s
roc_auc_score(train):           0.004957 s
predict_proba(X_test):          0.0809 s
roc_auc_score(test): 

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("et", ExtraTreesClassifier(
        n_estimators=600,
        max_depth=None,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        n_jobs=1,
        random_state=0
    ))
])

param_grid = {
    "et__n_estimators": [400, 800],
    "et__max_depth": [20, 50],
    "et__min_samples_split": [2, 10, 30],
    "et__min_samples_leaf": [5, 10],
    "et__max_features": ["sqrt", 0.2],
    "et__class_weight": [None, "balanced_subsample"],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)

grid = GridSearchCV(
    pipe,
    param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)
best = grid.best_estimator_

print("best estimator:", grid.best_estimator_)
print("best params:", grid.best_params_)
print("best CV auc:", grid.best_score_)

preds_train = best.predict_proba(X_train)[:, 1]
print("train auc-roc: ", roc_auc_score(y_train, preds_train))

preds_test = best.predict_proba(X_test)[:, 1]
print("test auc-roc: ", roc_auc_score(y_test, preds_test))

In [27]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("ada", AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=2),
        algorithm="SAMME",
        n_estimators=300,
        learning_rate=0.05,
        random_state=0
    ))
])

param_grid = {
    "ada__n_estimators": [100, 300, 600],
    "ada__learning_rate": [0.01, 0.05, 0.1, 0.3],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)

grid = GridSearchCV(
    pipe,
    param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)
best = grid.best_estimator_

print("best estimator:", grid.best_estimator_)
print("best params:", grid.best_params_)
print("best CV auc:", grid.best_score_)

preds_train = best.predict_proba(X_train)[:, 1]
print("train auc-roc: ", roc_auc_score(y_train, preds_train))

preds_test = best.predict_proba(X_test)[:, 1]
print("test auc-roc: ", roc_auc_score(y_test, preds_test))

Fitting 5 folds for each of 12 candidates, totalling 60 fits
best estimator: Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('ada',
                 AdaBoostClassifier(algorithm='SAMME',
                                    estimator=DecisionTreeClassifier(max_depth=2),
                                    learning_rate=0.05, n_estimators=300,
                                    random_state=0))])
best params: {'ada__learning_rate': 0.05, 'ada__n_estimators': 300}
best CV auc: 0.7239471546337062
train auc-roc:  0.7789539877038965
test auc-roc:  0.7523909602663839


In [26]:
# from xgboost import XGBClassifier
# pos_rate = float(np.mean(y_train))
# scale_pos_weight = (1 - pos_rate) / pos_rate   # ~8.55 for your data

# pipe = Pipeline([
#     ("imputer", SimpleImputer(strategy="median")),
#     ("xgb", XGBClassifier(
#         objective="binary:logistic",
#         eval_metric="auc",
#         tree_method="hist",        # fast on CPU
#         n_estimators=800,
#         learning_rate=0.05,
#         max_depth=4,
#         subsample=0.8,
#         colsample_bytree=0.4,      # good with many features
#         reg_lambda=1.0,
#         reg_alpha=0.0,
#         min_child_weight=1.0,
#         gamma=0.0,
#         scale_pos_weight=scale_pos_weight,
#         n_jobs=1,
#         random_state=0
#     ))
# ])

# param_grid = {
#     "xgb__max_depth": [3, 4, 5],
#     "xgb__learning_rate": [0.03, 0.05],
#     "xgb__n_estimators": [600, 1000],
#     "xgb__subsample": [0.7, 0.9],
#     "xgb__colsample_bytree": [0.2, 0.4, 0.6],
#     "xgb__min_child_weight": [1, 5, 10],
#     "xgb__reg_lambda": [1.0, 5.0],
# }

# grid = GridSearchCV(
#     pipe,
#     param_grid,
#     scoring="roc_auc",
#     cv=cv,
#     n_jobs=-1,
#     verbose=1
# )

# grid.fit(X_train, y_train)
# best = grid.best_estimator_

# print("best estimator:", grid.best_estimator_)
# print("best params:", grid.best_params_)
# print("best CV auc:", grid.best_score_)

# preds_train = best.predict_proba(X_train)[:, 1]
# print("train auc-roc: ", roc_auc_score(y_train, preds_train))

# preds_test = best.predict_proba(X_test)[:, 1]
# print("test auc-roc: ", roc_auc_score(y_test, preds_test))

# # best params: {'xgb__colsample_bytree': 0.2, 'xgb__learning_rate': 0.03, 'xgb__max_depth': 3, 'xgb__min_child_weight': 10, 'xgb__n_estimators': 600, 'xgb__reg_lambda': 5.0, 'xgb__subsample': 0.9}
# # best CV auc: 0.7287714573854669
# # train auc-roc:  0.9796115363874249
# # test auc-roc:  0.7492646070687312

Fitting 5 folds for each of 432 candidates, totalling 2160 fits
best estimator: Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('xgb',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=0.2, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric='auc',
                               feature_types=None, feature_weights=None,
                               gamma=0.0, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.03,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=3, max_leaves=None,
   

In [ ]:
# from lightgbm import LGBMClassifier

# pos_rate = float(np.mean(y_train))

# pipe = Pipeline([
#     ("imputer", SimpleImputer(strategy="median")),
#     ("lgbm", LGBMClassifier(
#         objective="binary",
#         metric="auc",
#         n_estimators=3000,
#         learning_rate=0.03,
#         num_leaves=31,
#         max_depth=-1,              # -1 means no limit
#         min_child_samples=20,
#         subsample=0.8,
#         colsample_bytree=0.4,
#         reg_lambda=1.0,
#         reg_alpha=0.0,
#         is_unbalance=True if pos_rate < 0.2 else False,  # or set scale_pos_weight instead
#         n_jobs=-1,
#         random_state=0
#     ))
# ])

# param_grid = {
#     "lgbm__num_leaves": [31, 63, 127],
#     "lgbm__learning_rate": [0.02, 0.03, 0.05],
#     "lgbm__n_estimators": [1500, 3000],
#     "lgbm__min_child_samples": [10, 20, 50],
#     "lgbm__subsample": [0.7, 0.9],
#     "lgbm__colsample_bytree": [0.2, 0.4, 0.6],
#     "lgbm__reg_lambda": [1.0, 5.0],
# }

# grid = GridSearchCV(
#     pipe,
#     param_grid,
#     scoring="roc_auc",
#     cv=cv,
#     n_jobs=-1,
#     verbose=1
# )

# grid.fit(X_train, y_train)
# best = grid.best_estimator_

# print("best estimator:", grid.best_estimator_)
# print("best params:", grid.best_params_)
# print("best CV auc:", grid.best_score_)

# preds_train = best.predict_proba(X_train)[:, 1]
# print("train auc-roc: ", roc_auc_score(y_train, preds_train))

# preds_test = best.predict_proba(X_test)[:, 1]
# print("test auc-roc: ", roc_auc_score(y_test, preds_test))

feature selection:

feature engineer:
- standard scaling
- pca
- the thing i did in 148
- correlation matrix
- l1

sampling:
- smote (for imbalanced data)
- near miss alg

models:
- logistic
- random forest
- xgboost
- lightgbm
- svm
- cnn
- clustering

accessment + latency (within each model)
- subgroup auc-roc
- auc-roc graphs
- latency reports
- 

pipeline:
- feat selection
- feat eng
- sampling
- model